---
title: "Advanced Polygenic Risk Scores (PRS)"
format:
  html:
    code-fold: false
    toc: true
  ipynb:
    toc: true
    number-sections: false
bibliography: references/references_6.bib
---

::: {.callout-note title="Important notes for this notebook" icon=false}

In `GWAS6` and `GWAS7`, we constructed Polygenic Risk Scores (PRS) using **Clumping + Thresholding (C+T)** (`PLINK` and `PRSice-2`), which greedily prunes SNPs in LD and uses unshrunk marginal GWAS effect sizes. In this 25-minute tutorial, we move to **advanced Bayesian and penalized PRS methods** that model all genome-wide SNPs jointly, using **SNP heritability ($h^2_{\text{SNP}}$)** from `GWAS8` to shrink noisy effect estimates. We compare the two primary paradigms using **`LDAK`**:
1. **Summary-statistics based PRS**: **`LDAK MegaPRS`** (`--mega-prs`, analogous to `LDpred2` and `PRS-CS`)
2. **Individual-level PRS**: **`LDAK-Bolt-Predict`** (`--bolt`) and **`LDAK-Elastic-Predict`** (`--elastic`)

## Learning outcomes

- **Contrast** heuristic C+T (`PRSice-2`) with joint Bayesian shrinkage methods (`MegaPRS`, `LDpred2`, `PRS-CS`, `LDAK-Predict`)
- **Explain** how per-SNP heritability estimates ($\mathbb{E}[h_j^2]$) regularize posterior SNP weights
- **Construct** summary-statistics based PRS using `LDAK MegaPRS` (`ldak --mega-prs`)
- **Construct** individual-level PRS using `LDAK-Bolt-Predict` (`ldak --bolt`) and `LDAK-Elastic-Predict` (`ldak --elastic`) and evaluate incremental $R^2$ in `R`

## How to make this notebook work

* In this notebook, we will use both `bash command line` and `R` programming languages. Remember to **change the kernel** whenever you transition from one language to the other (`Kernel --> Change Kernel`) indicated by the languages' images. We will first run `Bash` commands.  
  
<img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash kernel
 
:::

:::{.callout-note title="Some theory"}

### Intuition First: From an "On/Off Switch" (`PRSice-2`) to a "Smart Volume Knob" (Bayesian Shrinkage)

In `GWAS6` and `GWAS7`, we built Polygenic Risk Scores using **Clumping + Thresholding (`PRSice-2`)**, which behaves like a crude **on/off switch**:
- First, **LD clumping** throws away all correlated SNPs in an LD block except the single lead SNP—even if the region contains multiple causal variants (`GWAS10`).
- Second, **$p$-value thresholding** trusts the noisy GWAS effect size $\hat{\beta}_j$ at **100% volume** if $p_j < P_T$, and sets it to **0%** if $p_j \ge P_T$.

**Advanced Bayesian and penalized PRS methods** (`LDAK MegaPRS`, `LDpred2`, `PRS-CS`, `LDAK-Predict`) replace that on/off switch with a **smart volume knob (shrinkage)** across all genome-wide SNPs simultaneously: instead of throwing correlated SNPs away, they account for local LD ($R$) and use each SNP's **expected heritability ($h_j^2$ from `GWAS8`)** to gently dial down noisy effect estimates.

### Why Move Beyond Clumping + Thresholding (C+T)?

A linear Polygenic Score for individual $i$ is $PGS_i = \sum_{j=1}^M X_{ij} \tilde{\beta}_j$. Modern PRS methods infer **joint posterior effect sizes** $\tilde{\beta} = \mathbb{E}[b \mid \text{Data}, R, \hat{h}^2_{\text{SNP}}]$ across all genome-wide SNPs simultaneously. Under an infinitesimal ridge prior where SNP $j$ has expected heritability $h_j^2$, the optimal shrinkage estimator is:

$$\tilde{\beta} = \left( R + \frac{1}{N} \text{diag}\!\left(\frac{1}{h_j^2}\right) \right)^{-1} \hat{\beta}_{\text{GWAS}}$$

When $h_j^2$ or sample size $N$ is small, the penalty $1 / (N h_j^2)$ is large and strongly shrinks noisy $\hat{\beta}_j$ toward zero.

### Summary-Statistics vs. Individual-Level PRS Estimators

| Category | Method | Prior on SNP Effects ($b_j$) | Heritability Model | Input Data Required |
|---|---|---|---|---|
| **Summary-Stats** | **C+T (`PRSice-2`)** | Hard $p$-value threshold + LD clumping | None | GWAS sumstats + LD ref |
| **Summary-Stats** | **`LDpred2`** | Point-normal (spike-and-slab) with proportion causal $p$ | GCTA ($\alpha = -1$) | GWAS sumstats + LD matrix $R$ |
| **Summary-Stats** | **`PRS-CS`** | Continuous global-local Strawderman-Berger shrinkage | GCTA ($\alpha = -1$) | GWAS sumstats + pre-computed LD |
| **Summary-Stats** | **`LDAK MegaPRS`** (`--mega-prs`) | **Ridge (`LDpred-inf`), Bolt (`LDpred2`), Lasso, or BayesR** | **LDAK ($\alpha = -0.25$) / BLD-LDAK** | **GWAS sumstats + `LDAK` LD cors** |
| **Individual-Level** | **`LDAK-Bolt-Predict`** (`--bolt`) | Mixture of two Gaussians (large & small effects) | **LDAK ($\alpha = -0.25$)** | **Individual genotypes ($X$) & phenotype ($y$)** |
| **Individual-Level** | **`LDAK-Elastic-Predict`** (`--elastic`) | Mixture of Laplace ($L_1$) & Gaussian ($L_2$) | **LDAK ($\alpha = -0.25$)** | **Individual genotypes ($X$) & phenotype ($y$)** |

**When should you use Summary-Statistics vs. Individual-Level PRS?**
- **Summary-Statistics (`MegaPRS`, `LDpred2`, `PRS-CS`)**: Best when the largest available training data is a published mega-consortium GWAS ($N > 500\text{,000}$) where individual genotypes cannot be accessed.
- **Individual-Level (`LDAK-Bolt-Predict`, `LDAK-Elastic-Predict`)**: Best when you have direct access to biobank genotypes and phenotypes. Fitting $X$ and $y$ directly avoids summary-statistic approximation errors and LD reference panel mismatch ([Zhang et al., 2021](https://doi.org/10.1038/s41467-021-24485-y)).

> **Quick Concept Check (from Lecture 11)**:
> - *If a trait is driven by a few large-effect SNPs plus thousands of pure noise SNPs, which `MegaPRS` prior fits best?* **`--model bolt` (Spike-and-Slab)**, which silences noise SNPs near zero while keeping large causal signals loud.
> - *If you have $N = 400$ local genotypes and a published GWAS of $N = 250\text{,000}$, how should you train PRS weights?* Run **`MegaPRS` on the $N = 250\text{,000}$ summary stats + an External LD Panel** — discovery sample size $N$ is the dominant driver of PRS accuracy!

:::

# Summary-Statistics Based PRS with `LDAK MegaPRS`

Let's build a genome-wide polygenic score from the base GWAS summary statistics (`Data/Height.QC.gz`) using **`LDAK MegaPRS`** (`ldak --mega-prs`):
- `--calc-cors <prefix>`: Computes local SNP-by-SNP LD correlations within `--window-kb 1000` (1 Mb) windows on the reference panel (`Data/EUR.QC`).
- `--mega-prs <prefix>`: Estimates joint posterior SNP effect sizes (`.effects`) from `--summary`, `--cors`, and the per-SNP heritabilities `--ind-hers` (`sumher_ldak.ind.hers` from `GWAS8`), using `--cv-proportion 0.1` (pseudo-validation) to select the best shrinkage parameters.
  - `--model bolt`: Fits a Bayesian spike-and-slab prior (the `LDAK` counterpart to `LDpred2`).
  - `--model ridge`: Fits an infinitesimal Gaussian prior (the `LDAK` counterpart to `LDpred-inf`).
- `--calc-scores <prefix>`: Multiplies the posterior SNP weights (`--scorefile`) by each target individual's genotypes (`--power 0` means weights are already on the per-allele scale) and writes the individual polygenic scores to `<prefix>.profile`.

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash kernel.

Let's link the `Data` folder, create `Results/GWAS11`, and run `MegaPRS`:

In [ ]:
ln -sf ../Data
mkdir -p Results/GWAS11

# Prepare target phenotype and covariates for LDAK
awk 'NR>1 {print $1, $2, $3}' Data/EUR.height > Results/GWAS11/EUR.pheno
awk 'NR>1 {print $0}' Data/EUR.covariate > Results/GWAS11/EUR.covar

# Reuse Height.summaries and sumher_ldak.ind.hers from GWAS8 if present; otherwise create them
if [ -f Results/GWAS8/sumher_ldak.ind.hers ] && [ -f Results/GWAS8/Height.summaries ]; then
  cp Results/GWAS8/Height.summaries Results/GWAS11/Height.summaries
  cp Results/GWAS8/sumher_ldak.ind.hers Results/GWAS11/sumher_ldak.ind.hers
else
  gzip -dc Data/Height.QC.gz | awk '
  NR==1 {
    for (i=1; i<=NF; i++) col[$i]=i
    print "Predictor A1 A2 Direction P n"
    next
  }
  $col["MAF"] >= 0.01 && $col["INFO"] >= 0.8 && !seen[$col["SNP"]]++ {
    eff = ("OR" in col) ? log($col["OR"]) : $col["BETA"]
    dir = (eff >= 0) ? 1 : -1
    p   = $col["P"]
    n   = ("N" in col) ? $col["N"] : 253288
    if (p > 0 && p <= 1) print $col["SNP"], $col["A1"], $col["A2"], dir, p, n
  }' > Results/GWAS11/Height.summaries
  ldak --thin Results/GWAS11/thin --bfile Data/EUR.QC --window-prune 0.98 --window-kb 100
  awk '{print $1, 1}' Results/GWAS11/thin.in > Results/GWAS11/weights.thin
  ldak --calc-tagging Results/GWAS11/tag_ldak --bfile Data/EUR.QC --weights Results/GWAS11/weights.thin --power -0.25 --window-kb 1000 --save-matrix YES
  ldak --sum-hers Results/GWAS11/sumher_ldak --summary Results/GWAS11/Height.summaries --tagfile Results/GWAS11/tag_ldak.tagging --matrix Results/GWAS11/tag_ldak.matrix --check-sums NO
fi

# 1. Compute local LD correlation windows (1000 kb) for MegaPRS
ldak \
  --calc-cors Results/GWAS11/cors_eur \
  --bfile Data/EUR.QC \
  --window-kb 1000

# 2. Run MegaPRS with Bolt (LDpred2 spike-and-slab) and Ridge (LDpred-inf) priors
ldak \
  --mega-prs Results/GWAS11/megaprs_bolt \
  --model bolt \
  --ind-hers Results/GWAS11/sumher_ldak.ind.hers \
  --summary Results/GWAS11/Height.summaries \
  --cors Results/GWAS11/cors_eur \
  --cv-proportion 0.1 \
  --window-kb 1000 \
  --check-sums NO

ldak \
  --mega-prs Results/GWAS11/megaprs_ridge \
  --model ridge \
  --ind-hers Results/GWAS11/sumher_ldak.ind.hers \
  --summary Results/GWAS11/Height.summaries \
  --cors Results/GWAS11/cors_eur \
  --cv-proportion 0.1 \
  --window-kb 1000 \
  --check-sums NO

# 3. Score target individuals in Data/EUR.QC
ldak --calc-scores Results/GWAS11/prs_megaprs_bolt --bfile Data/EUR.QC --scorefile Results/GWAS11/megaprs_bolt.effects --power 0 --pheno Results/GWAS11/EUR.pheno
ldak --calc-scores Results/GWAS11/prs_megaprs_ridge --bfile Data/EUR.QC --scorefile Results/GWAS11/megaprs_ridge.effects --power 0 --pheno Results/GWAS11/EUR.pheno

# Individual-Level PRS with `LDAK-Bolt-Predict` and `LDAK-Elastic-Predict`

Now suppose we have direct access to individual-level genotypes $X$ and phenotypes $y$ in a biobank. Let's split `Data/EUR.QC` into an **80% Training set** (`train.keep`) and a **20% Hold-out Test set** (`test.keep`), check training SNP heritability (`--reml`), and train two individual-level whole-genome regression predictors on the training set (passing `--LOCO NO` so `LDAK` builds a single genome-wide prediction model (`*.effects`) rather than 22 chromosome-specific LOCO predictors):
- `--bolt <prefix>`: Fits individual-level **Bolt-Predict** (mixture of two Gaussians prior) using `--keep Results/GWAS11/train.keep`, `--ind-hers Results/GWAS11/sumher_ldak.ind.hers`, and `--LOCO NO`.
- `--elastic <prefix>`: Fits individual-level **Elastic-Predict** (mixture of Laplace $L_1$ and Gaussian $L_2$ priors) with `--LOCO NO`.

Finally, we project the trained weights onto the held-out **20% Test set** (`--keep Results/GWAS11/test.keep`) using `ldak --calc-scores`.

In [ ]:
# 1. Create an 80% Training / 20% Testing split
awk 'NR % 5 != 0 {print $1, $2}' Results/GWAS11/EUR.pheno > Results/GWAS11/train.keep
awk 'NR % 5 == 0 {print $1, $2}' Results/GWAS11/EUR.pheno > Results/GWAS11/test.keep

# 2. Estimate SNP heritability on the 80% Training set using REML
ldak --calc-kins-direct Results/GWAS11/kin_train --bfile Data/EUR.QC --keep Results/GWAS11/train.keep --power -0.25
ldak --reml Results/GWAS11/reml_train --grm Results/GWAS11/kin_train --pheno Results/GWAS11/EUR.pheno --covar Results/GWAS11/EUR.covar --keep Results/GWAS11/train.keep

# 3. Fit Individual-Level Bolt-Predict (--bolt) and Elastic-Predict (--elastic) on Training set
ldak \
  --bolt Results/GWAS11/ind_bolt \
  --bfile Data/EUR.QC \
  --pheno Results/GWAS11/EUR.pheno \
  --covar Results/GWAS11/EUR.covar \
  --keep Results/GWAS11/train.keep \
  --ind-hers Results/GWAS11/sumher_ldak.ind.hers \
  --cv-proportion 0.1 \
  --LOCO NO

ldak \
  --elastic Results/GWAS11/ind_elastic \
  --bfile Data/EUR.QC \
  --pheno Results/GWAS11/EUR.pheno \
  --covar Results/GWAS11/EUR.covar \
  --keep Results/GWAS11/train.keep \
  --ind-hers Results/GWAS11/sumher_ldak.ind.hers \
  --cv-proportion 0.1 \
  --LOCO NO

# 4. Project trained weights onto the held-out 20% Test set
ldak --calc-scores Results/GWAS11/prs_ind_bolt --bfile Data/EUR.QC --keep Results/GWAS11/test.keep --scorefile Results/GWAS11/ind_bolt.effects --power 0 --pheno Results/GWAS11/EUR.pheno
ldak --calc-scores Results/GWAS11/prs_ind_elastic --bfile Data/EUR.QC --keep Results/GWAS11/test.keep --scorefile Results/GWAS11/ind_elastic.effects --power 0 --pheno Results/GWAS11/EUR.pheno

### Evaluating PRS Accuracy in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

Just as in `GWAS6` and `GWAS7`, we evaluate polygenic score accuracy in `R` by computing the **incremental $R^2$** (`r2_full - r2_null`)—the additional phenotypic variance explained when adding the standardized PRS (`Profile_1`) to a baseline model containing only covariates (`Height ~ Sex + PC1 + ... + PC6`).

In [ ]:
# Setup to avoid long messages and plot on screen
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 8.5, repr.plot.height = 4.5)

suppressMessages(suppressWarnings(library(ggplot2)))

# Helper function to compute incremental R2 over Sex + PC1..PC6
# Note: ldak --calc-scores outputs .profile with columns ID1, ID2, Phenotype, Covariates, Profile_1
calc_incremental_r2 <- function(profile_file, pheno_df, covar_df) {
  prs <- read.table(profile_file, header = TRUE)
  colnames(prs)[1:2] <- c("FID", "IID")
  dat <- merge(pheno_df, covar_df, by = c("FID", "IID"))
  dat <- merge(dat, prs[, c("FID", "IID", "Profile_1")], by = c("FID", "IID"))
  dat$PRS_std <- as.numeric(scale(dat$Profile_1))
  
  null_fit <- lm(Height ~ Sex + PC1 + PC2 + PC3 + PC4 + PC5 + PC6, data = dat)
  full_fit <- lm(Height ~ PRS_std + Sex + PC1 + PC2 + PC3 + PC4 + PC5 + PC6, data = dat)
  
  list(
    inc_r2 = summary(full_fit)$r.squared - summary(null_fit)$r.squared,
    p_val  = summary(full_fit)$coefficients["PRS_std", "Pr(>|t|)"],
    data   = dat
  )
}

pheno <- read.table("Data/EUR.height", header = TRUE)
covar <- read.table("Data/EUR.covariate", header = TRUE)

res_mega_bolt  <- calc_incremental_r2("Results/GWAS11/prs_megaprs_bolt.profile", pheno, covar)
res_mega_ridge <- calc_incremental_r2("Results/GWAS11/prs_megaprs_ridge.profile", pheno, covar)
res_ind_bolt   <- calc_incremental_r2("Results/GWAS11/prs_ind_bolt.profile", pheno, covar)
res_ind_elas   <- calc_incremental_r2("Results/GWAS11/prs_ind_elastic.profile", pheno, covar)

cat("Summary-Stats MegaPRS (Bolt / LDpred2) Incremental R2:  ", round(res_mega_bolt$inc_r2, 4), "(p =", signif(res_mega_bolt$p_val, 3), ")\n")
cat("Summary-Stats MegaPRS (Ridge / LDpred-inf) Inc. R2:     ", round(res_mega_ridge$inc_r2, 4), "(p =", signif(res_mega_ridge$p_val, 3), ")\n")
cat("Individual-Level LDAK-Bolt (N_train=380) Inc. R2:       ", round(res_ind_bolt$inc_r2, 4), "(p =", signif(res_ind_bolt$p_val, 3), ")\n")
cat("Individual-Level LDAK-Elastic (N_train=380) Inc. R2:    ", round(res_ind_elas$inc_r2, 4), "(p =", signif(res_ind_elas$p_val, 3), ")\n")

:::{.callout-important title="Stop - Read - Solve"}

- **Q1. Create a bar plot comparing the incremental $R^2$ (`inc_r2`) of the four models (`MegaPRS-Bolt`, `MegaPRS-Ridge`, `Individual-Bolt`, and `Individual-Elastic`).**
- **Q2. Why do the summary-statistics `MegaPRS` models achieve much higher incremental $R^2$ than the individual-level models in our exercise dataset, whereas in the UK Biobank ($N = 400\text{,000}$) individual-level `LDAK-Bolt` / `LDAK-Elastic` outperform summary-statistics methods?**

Hint: Compare the discovery sample size of `Data/Height.QC.gz` ($N = 253\text{,288}$) against our 80% individual-level training split of `Data/EUR.QC` ($N_{\text{train}} \approx 380$).
:::

In [ ]:
# Write your code here


:::{.callout-tip title="Solution" collapse="true"}

In `R`, we compare the incremental $R^2$ across the summary-statistics and individual-level models:

In [ ]:
# Q1. Bar plot comparing incremental R2 across Summary-Stats and Individual-Level PRS models
bench_df <- data.frame(
  Method   = factor(c("MegaPRS (Bolt)", "MegaPRS (Ridge)", "LDAK-Bolt (Ind)", "LDAK-Elastic (Ind)"),
                    levels = c("MegaPRS (Bolt)", "MegaPRS (Ridge)", "LDAK-Bolt (Ind)", "LDAK-Elastic (Ind)")),
  Category = c("Summary Statistics (N=253k)", "Summary Statistics (N=253k)",
               "Individual-Level (N=380)", "Individual-Level (N=380)"),
  Inc_R2   = c(res_mega_bolt$inc_r2, res_mega_ridge$inc_r2,
               res_ind_bolt$inc_r2, res_ind_elas$inc_r2)
)

print(bench_df)

p_prs <- ggplot(bench_df, aes(x = Method, y = Inc_R2, fill = Category)) +
  geom_col(width = 0.6, col = "black") +
  geom_text(aes(label = round(Inc_R2, 4)), vjust = -0.4, size = 4) +
  scale_fill_manual(values = c("Summary Statistics (N=253k)" = "steelblue",
                               "Individual-Level (N=380)" = "tomato")) +
  theme_bw(base_size = 13) +
  ylim(0, max(bench_df$Inc_R2) * 1.18) +
  labs(
    title = "Incremental R2: Summary-Statistics MegaPRS vs. Individual-Level LDAK-Predict",
    x = "PRS Estimator",
    y = expression("Incremental " ~ R^2 ~ " over Covariates"),
    fill = "Training Paradigm"
  )

show(p_prs)

- **Q1**: `MegaPRS (Bolt)` and `MegaPRS (Ridge)` explain a substantial fraction of height variance over covariates in `Data/EUR.QC`, with the spike-and-slab `Bolt` model slightly outperforming the infinitesimal `Ridge` model.
- **Q2**: The theoretical accuracy of a polygenic predictor scales with $N h^2_{\text{SNP}} / (N h^2_{\text{SNP}} + M_e)$, where $N$ is the training sample size. In this tutorial, `Data/Height.QC.gz` comes from a consortium GWAS of **$N = 253\text{,288}$ individuals**, whereas our 80% training split of `Data/EUR.QC` has only **$N_{\text{train}} \approx 380$ individuals**. With $N = 380$ and $M \approx 500\text{,000}$ SNPs, the noise in individual-level estimation is huge, so `reml_train` and `--bolt`/`--elastic` strongly shrink weights toward zero. When sample sizes are matched at biobank scale ($N > 100\text{,000}$), individual-level `LDAK-Bolt` and `LDAK-Elastic` consistently outperform summary-statistics estimators because they access the exact individual-level LD structure without summary-statistic approximation errors.

:::

---

### Conclusion

In this tutorial, we have brought together the entire advanced course pipeline: combining **per-SNP heritability priors** (`sumher_ldak.ind.hers` from `GWAS8`), **GWAS summary statistics** (from `GWAS5`/`GWAS9`), and **external LD reference panel correlations** (`ldak --calc-cors`, mirroring local LD deconvolution in `GWAS10`) to build summary-statistics based **`LDAK MegaPRS`** (`--mega-prs`), and comparing it with individual-level whole-genome regression (**`LDAK-Predict`**: `--bolt` and `--elastic`, using the same engine as `GWAS9` Step 1).

:::{.callout-note title="Other take-home messages"}

- **Joint Shrinkage Outperforms Clumping + Thresholding**: Methods like `LDAK MegaPRS`, `LDpred2`, and `PRS-CS` avoid discarding correlated variants and shrink noisy GWAS estimates using SNP heritability ($h^2_{\text{SNP}}$).
- **Match the Estimator to Your Available Data**: Use `ldak --mega-prs` when leveraging large consortium GWAS summary statistics, and use individual-level `ldak --bolt` or `ldak --elastic` when training directly on large biobank cohorts.
- **Heritability Ties the Entire Advanced Pipeline Together**: From estimating $h^2_{\text{SNP}}$ (`GWAS8`), to conditioning on LOCO polygenic predictions in mixed-model GWAS (`GWAS9`), calibrating causal effect priors in fine-mapping (`GWAS10`), and shrinking polygenic weights (`GWAS11`), realistic heritability modeling is the foundation of modern statistical genetics.

:::